In [73]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [74]:
from pathlib import Path
import pandas as pd
import html
import unicodedata
import re
from bs4 import BeautifulSoup

project_dir = Path("/content/drive/MyDrive/Colab Notebooks/DATA602-Project")

dataset_path = project_dir / "data/amazon_reviews_us_Digital_Music_Purchase_v1_00.tsv"

In [75]:
df = pd.read_csv(
    dataset_path,
    sep="\t",
    on_bad_lines="skip",
    low_memory=True
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1681484, 15)


,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date
0,US,10293599,R14LVB34Z2Z53I,B000YMOQZY,262935067,Knockin' On Heaven's Door,Digital_Music_Purchase,5.0,0.0,0.0,N,Y,favorites,Great rendition. Great song,2015-08-31
1,US,6087195,R2M0R9CWPC82LT,B00ISAEC80,452318038,Flawless,Digital_Music_Purchase,5.0,0.0,0.0,N,Y,Five Stars,A good music to listen.,2015-08-31
2,US,33717063,RQNQI2R3BM52A,B00VBB6AVU,675969129,Scandal of Grace,Digital_Music_Purchase,4.0,0.0,0.0,N,Y,Good,Not as good as the original but I needed to he...,2015-08-31
3,US,14948020,RTUIX1135HUAB,B008A5POJO,282925893,I Surrender (Live),Digital_Music_Purchase,5.0,0.0,0.0,N,Y,Wonderful lyrics! Makes you feel like praying,Time to worship and surrender with this song. ...,2015-08-31
4,US,21694522,RE3QSAT8TL010,B014CW2RN4,109077793,We Are Young (Jersey Club),Digital_Music_Purchase,4.0,0.0,0.0,N,Y,Hot remix but took short,So this should be longer....but love this song:),2015-08-31


In [37]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nMissing values:")
print(df.isnull().sum())

Rows: 1681484
Columns: 15

Missing values:
marketplace            0
customer_id            0
review_id              0
product_id             0
product_parent         0
product_title          0
product_category       0
star_rating            2
helpful_votes          2
total_votes            2
vine                   2
verified_purchase      2
review_headline       59
review_body          114
review_date           76
dtype: int64


## 1. Data Preprocessing

The Amazon Digital Music Purchase Reviews dataset contains **1,681,484 records and 15 original features**.

The following preprocessing steps were performed:

1. **Data Type Conversion:** Converted identifiers to strings, ratings and vote counts to nullable integers, review dates to datetime, and categorical attributes to appropriate data types.
2. **Boolean Transformation:** Converted `verified_purchase` and `vine` values (Y/N) into boolean features (`is_verified` and `is_vine`).
3. **Temporal Feature Extraction:** Extracted `review_year` and `review_month` to support time-based analysis.
4. **Text Feature Extraction:** Created `review_word_count` and `review_char_count` to measure review length.
5. **Helpfulness Ratio:** Calculated the proportion of helpful votes using `helpful_votes / total_votes`, leaving the ratio undefined when total votes equal zero.

### Preprocessing Summary

- **Original Features:** 15
- **New Features:** 7
- **Total Features:** 22
- **Records Retained During Preprocessing:** 1,681,484

Missing values were preserved for the subsequent data cleaning stage. Malformed records encountered during initial TSV import were skipped using `on_bad_lines="skip"`.

These transformations prepare the dataset for data cleaning, exploratory analysis, and statistical testing.

In [38]:
df_preprocessed = df.copy()

print("\nColumn Information:")
df_preprocessed.info()


Column Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1681484 entries, 0 to 1681483
Data columns (total 15 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   marketplace        1681484 non-null  object 
 1   customer_id        1681484 non-null  int64  
 2   review_id          1681484 non-null  object 
 3   product_id         1681484 non-null  object 
 4   product_parent     1681484 non-null  int64  
 5   product_title      1681484 non-null  object 
 6   product_category   1681484 non-null  object 
 7   star_rating        1681482 non-null  float64
 8   helpful_votes      1681482 non-null  float64
 9   total_votes        1681482 non-null  float64
 10  vine               1681482 non-null  object 
 11  verified_purchase  1681482 non-null  object 
 12  review_headline    1681425 non-null  object 
 13  review_body        1681370 non-null  object 
 14  review_date        1681408 non-null  object 
dtypes: float64(

In [39]:
id_cols = [
    "customer_id", "review_id", "product_id", "product_parent"
]

for cols in id_cols:
    df_preprocessed[cols] = df_preprocessed[cols].astype("string")

df_preprocessed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1681484 entries, 0 to 1681483
Data columns (total 15 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   marketplace        1681484 non-null  object 
 1   customer_id        1681484 non-null  string 
 2   review_id          1681484 non-null  string 
 3   product_id         1681484 non-null  string 
 4   product_parent     1681484 non-null  string 
 5   product_title      1681484 non-null  object 
 6   product_category   1681484 non-null  object 
 7   star_rating        1681482 non-null  float64
 8   helpful_votes      1681482 non-null  float64
 9   total_votes        1681482 non-null  float64
 10  vine               1681482 non-null  object 
 11  verified_purchase  1681482 non-null  object 
 12  review_headline    1681425 non-null  object 
 13  review_body        1681370 non-null  object 
 14  review_date        1681408 non-null  object 
dtypes: float64(3), object(8), string

In [40]:
numeric_cols = [
    "star_rating",
    "helpful_votes",
    "total_votes"
]

for col in numeric_cols:
    df_preprocessed[col] = pd.to_numeric(
        df_preprocessed[col],
        errors="coerce"
    ).astype("Int64")

df_preprocessed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1681484 entries, 0 to 1681483
Data columns (total 15 columns):
 #   Column             Non-Null Count    Dtype 
---  ------             --------------    ----- 
 0   marketplace        1681484 non-null  object
 1   customer_id        1681484 non-null  string
 2   review_id          1681484 non-null  string
 3   product_id         1681484 non-null  string
 4   product_parent     1681484 non-null  string
 5   product_title      1681484 non-null  object
 6   product_category   1681484 non-null  object
 7   star_rating        1681482 non-null  Int64 
 8   helpful_votes      1681482 non-null  Int64 
 9   total_votes        1681482 non-null  Int64 
 10  vine               1681482 non-null  object
 11  verified_purchase  1681482 non-null  object
 12  review_headline    1681425 non-null  object
 13  review_body        1681370 non-null  object
 14  review_date        1681408 non-null  object
dtypes: Int64(3), object(8), string(4)
memory usage: 1

In [42]:
df_preprocessed["review_date"] = pd.to_datetime(
    df_preprocessed["review_date"],
    errors="coerce"
)

In [43]:
for col in ["marketplace", "product_category"]:
    df_preprocessed[col] = df_preprocessed[col].astype("category")

In [44]:
for col in [
    "product_title","review_headline","review_body"
]:
    df_preprocessed[col] = (
        df_preprocessed[col].astype("string")
    )

In [45]:
df_preprocessed["is_verified"] = (
    df_preprocessed["verified_purchase"]
    .map({"Y": True, "N": False})
    .astype("boolean")
)

df_preprocessed["is_vine"] = (
    df_preprocessed["vine"]
    .map({"Y": True, "N": False})
    .astype("boolean")
)

In [46]:
df_preprocessed["review_year"] = (
    df_preprocessed["review_date"].dt.year
)

df_preprocessed["review_month"] = (
    df_preprocessed["review_date"].dt.month
)

In [47]:
df_preprocessed["review_word_count"] = (
    df_preprocessed["review_body"].str.count(r"\S+")
)

df_preprocessed["review_char_count"] = (
    df_preprocessed["review_body"].str.len()
)

In [48]:
df_preprocessed["helpfulness_ratio"] = (
    df_preprocessed["helpful_votes"] /
    df_preprocessed["total_votes"].where(
        df_preprocessed["total_votes"] > 0
    )
)

In [49]:
print(df_preprocessed.shape, "\n")
print(df_preprocessed.info())

(1681484, 22) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1681484 entries, 0 to 1681483
Data columns (total 22 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   marketplace        1681484 non-null  category      
 1   customer_id        1681484 non-null  string        
 2   review_id          1681484 non-null  string        
 3   product_id         1681484 non-null  string        
 4   product_parent     1681484 non-null  string        
 5   product_title      1681484 non-null  string        
 6   product_category   1681484 non-null  category      
 7   star_rating        1681482 non-null  Int64         
 8   helpful_votes      1681482 non-null  Int64         
 9   total_votes        1681482 non-null  Int64         
 10  vine               1681482 non-null  object        
 11  verified_purchase  1681482 non-null  object        
 12  review_headline    1681425 non-null  string        
 13  review_body

In [50]:
import os

project_dir = "/content/drive/MyDrive/Colab Notebooks/DATA602-Project"

os.chdir(project_dir)
print(os.getcwd())

/content/drive/.shortcut-targets-by-id/1f5UsS2DAecT3XX9RUR1zTFWLeruZYSz3/DATA602-Project


In [51]:
df_preprocessed.head()

,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,...,review_headline,review_body,review_date,is_verified,is_vine,review_year,review_month,review_word_count,review_char_count,helpfulness_ratio
0,US,10293599,R14LVB34Z2Z53I,B000YMOQZY,262935067,Knockin' On Heaven's Door,Digital_Music_Purchase,5,0,0,...,favorites,Great rendition. Great song,2015-08-31,True,False,2015.0,8.0,4,29,<NA>
1,US,6087195,R2M0R9CWPC82LT,B00ISAEC80,452318038,Flawless,Digital_Music_Purchase,5,0,0,...,Five Stars,A good music to listen.,2015-08-31,True,False,2015.0,8.0,5,23,<NA>
2,US,33717063,RQNQI2R3BM52A,B00VBB6AVU,675969129,Scandal of Grace,Digital_Music_Purchase,4,0,0,...,Good,Not as good as the original but I needed to he...,2015-08-31,True,False,2015.0,8.0,22,103,<NA>
3,US,14948020,RTUIX1135HUAB,B008A5POJO,282925893,I Surrender (Live),Digital_Music_Purchase,5,0,0,...,Wonderful lyrics! Makes you feel like praying,Time to worship and surrender with this song. ...,2015-08-31,True,False,2015.0,8.0,15,93,<NA>
4,US,21694522,RE3QSAT8TL010,B014CW2RN4,109077793,We Are Young (Jersey Club),Digital_Music_Purchase,4,0,0,...,Hot remix but took short,So this should be longer....but love this song:),2015-08-31,True,False,2015.0,8.0,8,48,<NA>


In [52]:
df_preprocessed.dtypes

,0
marketplace,category
customer_id,string[python]
review_id,string[python]
product_id,string[python]
product_parent,string[python]
product_title,string[python]
product_category,category
star_rating,Int64
helpful_votes,Int64
total_votes,Int64


# 2. Data Cleaning

Data Cleaning is the process of identifying and correcting or removing inaccurate, incomplete, duplicate, inconsistent, or irrelevant data to improve data quality so that it is ready for model training.

The following data cleaning steps were peformed:


* **Data Quality Assessment** - Examining column data types, missing-value counts, missing-value percentages, and unique-value counts to assess data quality.

* **Duplicate Detection and Removal** - Identifying and removing duplicate records, if any, while retaining the first occurrence.

* **Missing Value Handling and Rating Validation** - Identifying and removing records with missing or blank review text and missing star ratings. Additionally, checking for invalid star ratings outside the valid range of 1–5.

* **HTML Formatting Inspection and Cleaning** - Identifying and removing HTML tags and decoding HTML entities in review text to ensure clean input for sentiment analysis.

* **Final Data Validation** - Performing final quality checks to ensure the cleaned dataset contains valid ratings, non-empty review text, and no exact duplicate records.

Before data cleaning, total records - **1,681,484**

After data cleaning, total recods - **1,681,368**








In [53]:
# Inspecting column data types, missing-value counts, missing-value percentages, and unique-value counts

column_summary = pd.DataFrame({
    "Column": df_preprocessed.columns,
    "Data Type": df_preprocessed.dtypes.values,
    "Missing Count": df_preprocessed.isna().sum().values,
    "Missing %": (
        df_preprocessed.isna().mean().values * 100
    ).round(2),
    "Unique Values": df_preprocessed.nunique().values
})

display(column_summary)



,Column,Data Type,Missing Count,Missing %,Unique Values
0,marketplace,category,0,0.00,1
1,customer_id,string[python],0,0.00,796587
2,review_id,string[python],0,0.00,1681484
3,product_id,string[python],0,0.00,699450
4,product_parent,string[python],0,0.00,644045
5,product_title,string[python],0,0.00,499915
6,product_category,category,0,0.00,3
7,star_rating,Int64,2,0.00,5
8,helpful_votes,Int64,2,0.00,201
9,total_votes,Int64,2,0.00,235


In [72]:
# Identifying exact duplicate rows
print("Exact duplicate rows:", df_preprocessed.duplicated().sum())

# Removing duplicate rows while retaining the first occurrence
rows_before = len(df_preprocessed)

df_preprocessed = df_preprocessed.drop_duplicates(
    keep="first"
).reset_index(drop=True)

# Calculating the number of duplicate rows removed
rows_after = len(df_preprocessed)

print("Duplicate rows removed:", rows_before - rows_after)
print("Remaining rows:", rows_after)

Exact duplicate rows: 0
Duplicate rows removed: 0
Remaining rows: 1681368


In [55]:
# Identifying missing values in review text and star ratings, which are essential for sentiment classification

# Removing records with missing review text or star ratings

rows_before = len(df_preprocessed)

df_preprocessed = df_preprocessed.dropna(
    subset=["review_body", "star_rating"]
).copy()


In [54]:
# Removing records containing only whitespace in the review body

df_preprocessed = df_preprocessed[
    df_preprocessed["review_body"].str.strip().ne("")
].copy()

In [57]:
# Validating star ratings to ensure they are suitable for sentiment labeling and subsequent model evaluation

print(
    df_preprocessed["star_rating"]
    .value_counts(dropna=False)
    .sort_index()
)

star_rating
1      48771
2      24927
3      62149
4     206455
5    1339068
Name: count, dtype: Int64


In [58]:
# Inspecting review text for HTML tags and encoded entities inherited from the original web-based source

TAG = re.compile(
    r"</?[a-z][^>]*>",
    re.IGNORECASE
)

ENTITY = re.compile(
    r"&(?:[a-z][a-z0-9]+|#\d+|#x[0-9a-f]+);",
    re.IGNORECASE
)

review_text = df_preprocessed["review_body"].astype("string")

html_tags = review_text.str.contains(
    TAG,
    na=False
)

html_entities = review_text.str.contains(
    ENTITY,
    na=False
)

print("Reviews containing HTML-like tags:", html_tags.sum())
print("Reviews containing HTML entities:", html_entities.sum())

Reviews containing HTML-like tags: 147426
Reviews containing HTML entities: 89495


In [64]:
# Cleaning review body column from these tags

# Creating a function

def clean_text(value):

    # Handle missing text
    if pd.isna(value):
        return ""

    # Convert to string and decode HTML entities
    text = html.unescape(str(value))

    # Remove HTML tags when detected
    if TAG.search(text):
        text = BeautifulSoup(
            text,
            "html.parser"
        ).get_text(" ")

    # Normalize Unicode
    text = unicodedata.normalize("NFC", text)

    # Replace control characters with spaces
    text = "".join(
        " " if (
            unicodedata.category(char) in ("Cc", "Cf")
            and char not in ("\n", "\t")
        ) else char
        for char in text
    )

    # Remove unnecessary whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text



In [65]:
# Applying cleaning function

df_preprocessed["clean_review"] = (
    df_preprocessed["review_body"]
    .map(clean_text)
)

print("Text cleaning completed successfully!")

# Identify reviews where cleaning made changes
changed_reviews = df_preprocessed[
    df_preprocessed["review_body"].fillna("") !=
    df_preprocessed["clean_review"]
]

print("Total reviews modified:", len(changed_reviews))


Text cleaning completed successfully!
Total reviews modified: 621270


In [69]:
# Identifying reviews that become empty after removing HTML tags and decoding HTML entities

empty_reviews = df_preprocessed[
    df_preprocessed["clean_review"].str.len() == 0
]

print("Total empty reviews:", len(empty_reviews))

display(
    empty_reviews[["review_body", "clean_review"]].head(20)
)

Total empty reviews: 2


,review_body,clean_review
155831,&#60;insert something deep here&#62;,
203600,...,


In [70]:
# Removing missing, empty, or whitespace-only cleaned reviews to ensure valid text inputs for model

rows_before = len(df_preprocessed)

df_preprocessed = df_preprocessed[
    df_preprocessed["clean_review"].notna() &
    df_preprocessed["clean_review"].str.strip().ne("")
].copy()

print("Rows removed:", rows_before - len(df_preprocessed))
print("Remaining rows:", len(df_preprocessed))

Rows removed: 2
Remaining rows: 1681368


Final cleaning validation

In [71]:
# Reset index
df_preprocessed = df_preprocessed.reset_index(drop=True)

# Essential fields must be present
assert df_preprocessed["clean_review"].notna().all()
assert df_preprocessed["star_rating"].notna().all()

# No blank review text
assert df_preprocessed["clean_review"].str.len().gt(0).all()

# Ratings must be valid
assert df_preprocessed["star_rating"].isin(
    [1, 2, 3, 4, 5]
).all()


# No fully duplicated rows
assert df_preprocessed.duplicated().sum() == 0

print("All essential cleaning checks passed!")
print("Final cleaned shape:", df_preprocessed.shape)

All essential cleaning checks passed!
Final cleaned shape: (1681368, 23)
